# Transaction data quality

This notebook analyses data quality problems in a deliberately corrupted copy of the transactions dataset

## Objectives

- Create a reproducible dataset containing known data quality problems
- Detect missing values
- Identify duplicate rows and duplicate transaction identifiers
- Detect invalid numeric values and dates
- Find inconsistent categorical values
- Validate account and merchant references
- Prepare the dataset for cleaning

In [28]:
from pathlib import Path
import pandas as pd

In [29]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

project_root

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence')

In [30]:
raw_transactions_path = (project_root / "data" / "raw" / "transactions.csv")
dirty_data_directory = (project_root / "data" / "dirty")
dirty_transactions_path = (dirty_data_directory / "transactions.csv")

print(f"Raw file: {raw_transactions_path}")
print(f"Dirty directory: {dirty_data_directory}")
print(f"Dirty file: {dirty_transactions_path}")

Raw file: c:\Users\Focus\Documents\GitHub\transaction-intelligence\data\raw\transactions.csv
Dirty directory: c:\Users\Focus\Documents\GitHub\transaction-intelligence\data\dirty
Dirty file: c:\Users\Focus\Documents\GitHub\transaction-intelligence\data\dirty\transactions.csv


In [31]:
dirty_data_directory.mkdir(parents = True, exist_ok = True)

In [32]:
dirty_data_directory.exists()

True

## Loading original dataset

The original transactions file is preserved unchanged; I create a separate copy before introducing any data quality problems

In [33]:
clean_transactions = pd.read_csv(raw_transactions_path)
clean_transactions.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
0,T0001,A001,M001,2025-01-03 08:15:00,34.80,EUR,Purchase,Completed
1,T0002,A003,M002,2025-01-04 10:42:00,4.20,EUR,Purchase,Completed
2,T0003,A005,M003,2025-01-05 19:10:00,18.99,EUR,Purchase,Completed
3,T0004,A006,M004,2025-01-06 21:35:00,249.90,EUR,Purchase,Completed
4,T0005,A008,M005,2025-01-08 12:05:00,62.40,EUR,Purchase,Completed


In [34]:
clean_transactions.shape

(60, 8)

In [35]:
dirty_transactions = clean_transactions.copy(deep = True)

In [36]:
dirty_transactions.shape

(60, 8)

In [37]:
dirty_transactions["amount"] =  (
    dirty_transactions["amount"].astype("string")
)

In [38]:
dirty_transactions.dtypes

transaction_id         str
account_id             str
merchant_id            str
timestamp              str
amount              string
currency               str
transaction_type       str
status                 str
dtype: object

In [39]:
# Here I will introduce some data quality issues to the transactions dataset

dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0007", "amount"] = pd.NA
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0028", "currency"] = pd.NA
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0018", "amount"] = "not_available"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0040", "amount"] = "-68.20"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0012", "timestamp"] = "2025-02-30 09:45:00"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0032", "timestamp"] = "14/02/2025 21:10"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0014", "currency"] = "usd"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0039", "transaction_type"] = "purchase"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0035", "status"] = "declined "
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0057", "status"] = "Complete"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0051","account_id"] = "A999"
dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0054","merchant_id"] = "M999"

exact_duplicate = dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0022"].copy()
dirty_transactions = pd.concat([dirty_transactions, exact_duplicate], ignore_index= True)


In [40]:
dirty_transactions.loc[dirty_transactions["transaction_id"].isin(["T0007", "T0028","T0018", "T0040"])]

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
6,T0007,A011,M007,2025-01-10 18:50:00,<NA>,EUR,Purchase,Completed
17,T0018,A003,M004,2025-01-26 12:44:00,not_available,EUR,Purchase,Completed
27,T0028,A002,M014,2025-02-09 14:30:00,9.8,NaN,Purchase,Completed
39,T0040,A014,M005,2025-02-27 12:10:00,-68.20,EUR,Purchase,Completed


In [41]:
dirty_transactions.shape

(61, 8)

In [42]:
duplicate_identifier = dirty_transactions.loc[dirty_transactions["transaction_id"] == "T0045"].copy()
duplicate_identifier["merchant_id"] = "M010"
duplicate_identifier["amount"] = "99.99"
dirty_transactions = pd.concat([dirty_transactions, duplicate_identifier], ignore_index= True)

In [43]:
dirty_transactions.shape

(62, 8)

In [44]:
# Save corrupted dataset
dirty_transactions.to_csv(dirty_transactions_path, index = False)

In [45]:
dirty_transactions_path.exists()

True

In [49]:
transactions_dirty = pd.read_csv(dirty_transactions_path)

In [50]:
transactions_dirty.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
0,T0001,A001,M001,2025-01-03 08:15:00,34.8,EUR,Purchase,Completed
1,T0002,A003,M002,2025-01-04 10:42:00,4.2,EUR,Purchase,Completed
2,T0003,A005,M003,2025-01-05 19:10:00,18.99,EUR,Purchase,Completed
3,T0004,A006,M004,2025-01-06 21:35:00,249.9,EUR,Purchase,Completed
4,T0005,A008,M005,2025-01-08 12:05:00,62.4,EUR,Purchase,Completed


In [51]:
transactions_dirty.shape

(62, 8)

In [52]:
transactions_dirty.dtypes

transaction_id      str
account_id          str
merchant_id         str
timestamp           str
amount              str
currency            str
transaction_type    str
status              str
dtype: object

# Initial data quality audit

I will now inspect the corrupted dataset without modifying or cleaning it

The goal is to identify and quantify each type of problem before deciding how it should be handled

In [54]:
transactions_dirty.info()

<class 'pandas.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   transaction_id    62 non-null     str  
 1   account_id        62 non-null     str  
 2   merchant_id       62 non-null     str  
 3   timestamp         62 non-null     str  
 4   amount            61 non-null     str  
 5   currency          61 non-null     str  
 6   transaction_type  62 non-null     str  
 7   status            62 non-null     str  
dtypes: str(8)
memory usage: 4.0 KB


In [55]:
transactions_dirty.head()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
0,T0001,A001,M001,2025-01-03 08:15:00,34.8,EUR,Purchase,Completed
1,T0002,A003,M002,2025-01-04 10:42:00,4.2,EUR,Purchase,Completed
2,T0003,A005,M003,2025-01-05 19:10:00,18.99,EUR,Purchase,Completed
3,T0004,A006,M004,2025-01-06 21:35:00,249.9,EUR,Purchase,Completed
4,T0005,A008,M005,2025-01-08 12:05:00,62.4,EUR,Purchase,Completed


In [56]:
transactions_dirty.tail()

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
57,T0058,A002,M011,2025-03-27 19:35:00,76.0,EUR,Purchase,Completed
58,T0059,A006,M004,2025-03-29 23:15:00,459.0,EUR,Purchase,Declined
59,T0060,A001,M003,2025-03-31 18:55:00,19.99,EUR,Refund,Completed
60,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed
61,T0045,A006,M010,2025-03-06 11:05:00,99.99,EUR,Purchase,Completed


In [ ]:
# Find missing values
missing_values = (transactions_dirty.isna().sum().sort_values(ascending = False))
missing_values

currency            1
amount              1
transaction_id      0
account_id          0
timestamp           0
merchant_id         0
transaction_type    0
status              0
dtype: int64

In [58]:
transactions_dirty.loc[transactions_dirty.isna().any(axis = 1)]

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
6,T0007,A011,M007,2025-01-10 18:50:00,NaN,EUR,Purchase,Completed
27,T0028,A002,M014,2025-02-09 14:30:00,9.8,NaN,Purchase,Completed


In [59]:
# Find duplicated rows
transactions_dirty.duplicated().sum()

np.int64(1)

In [61]:
exact_duplicated_rows = transactions_dirty.loc[transactions_dirty.duplicated(keep = False)]
exact_duplicated_rows 

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
21,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed
60,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed


In [62]:
# Find duplicated identificators

duplicated_id_mask = (transactions_dirty["transaction_id"].duplicated(keep = False))

duplicated_transaction_ids = (transactions_dirty.loc[duplicated_id_mask].sort_values("transaction_id"))
duplicated_transaction_ids

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
21,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed
60,T0022,A011,M002,2025-02-02 11:25:00,5.6,EUR,Purchase,Completed
44,T0045,A006,M012,2025-03-06 11:05:00,26.4,EUR,Purchase,Completed
61,T0045,A006,M010,2025-03-06 11:05:00,99.99,EUR,Purchase,Completed


In [63]:
duplicated_transaction_ids["transaction_id"].nunique()

2

In [64]:
# Examine categories

transactions_dirty["currency"].value_counts(dropna= False)

currency
EUR    58
USD     2
usd     1
NaN     1
Name: count, dtype: int64

In [65]:
transactions_dirty["status"].value_counts(dropna=False)

status
Completed    58
Declined      2
declined      1
Complete      1
Name: count, dtype: int64

In [66]:
transactions_dirty["transaction_type"].value_counts(dropna = False)

transaction_type
Purchase    60
purchase     1
Refund       1
Name: count, dtype: int64

In [67]:
amount_numeric = pd.to_numeric(transactions_dirty["amount"], errors= "coerce")

invalid_numeric_amounts = transactions_dirty.loc[amount_numeric.isna(), ["transaction_id", "amount"]]
invalid_numeric_amounts

,transaction_id,amount
6,T0007,NaN
17,T0018,not_available


In [68]:
# Find amounts that are not positive

non_positive_amounts = transactions_dirty.loc[amount_numeric.notna() & (amount_numeric <= 0), ["transaction_id", "amount"]]
non_positive_amounts

,transaction_id,amount
39,T0040,-68.20


In [ ]:
# Temporal version for datetime conversion

timestamp_parsed = pd.to_datetime(transactions_dirty["timestamp"], format = "mixed", errors = "coerce")

# Exposure of invalid dates

invalid_timestamps = transactions_dirty.loc[timestamp_parsed.isna(), ["transaction_id", "timestamp"]]
invalid_timestamps

,transaction_id,timestamp
11,T0012,2025-02-30 09:45:00


In [72]:
len("2025-02-30 09:45:00")

19

In [73]:
non_standard_timestamps = transactions_dirty.loc[transactions_dirty["timestamp"].str.len().ne(19)]
non_standard_timestamps

,transaction_id,account_id,merchant_id,timestamp,amount,currency,transaction_type,status
31,T0032,A001,M010,14/02/2025 21:10,15.5,EUR,Purchase,Completed


In [74]:
# Now I will validate accounts and merchants datasets

accounts_path = (project_root / "data" / "raw"/ "accounts.csv")

merchants_path = (project_root / "data" / "raw" / "merchants.csv")

accounts = pd.read_csv(accounts_path)
merchants = pd.read_csv(merchants_path)

In [76]:
# Find non existant accounts

invalid_account_references = transactions_dirty.loc[~transactions_dirty["account_id"].isin(accounts["account_id"]),
                                                    ["transaction_id", "account_id"]]

invalid_account_references

,transaction_id,account_id
50,T0051,A999


In [77]:
# Find non existant merchants

invalid_merchant_references = transactions_dirty.loc[~transactions_dirty["merchant_id"].isin(merchants["merchant_id"]),
                                                    ["transaction_id", "merchant_id"]]

invalid_merchant_references

,transaction_id,merchant_id
53,T0054,M999


In [78]:
# Summary

quality_report = pd.Series(
    {
        "number_of_rows": len(
            transactions_dirty
        ),
        "missing_values": int(
            transactions_dirty
            .isna()
            .sum()
            .sum()
        ),
        "exact_duplicate_rows": int(
            transactions_dirty
            .duplicated()
            .sum()
        ),
        "duplicated_transaction_ids": int(
            duplicated_transaction_ids[
                "transaction_id"
            ].nunique()
        ),
        "invalid_or_missing_amounts": int(
            amount_numeric
            .isna()
            .sum()
        ),
        "non_positive_amounts": int(
            len(non_positive_amounts)
        ),
        "invalid_timestamps": int(
            timestamp_parsed
            .isna()
            .sum()
        ),
        "non_standard_timestamps": int(
            len(non_standard_timestamps)
        ),
        "invalid_account_references": int(
            len(invalid_account_references)
        ),
        "invalid_merchant_references": int(
            len(invalid_merchant_references)
        ),
    },
    name="count",
)

quality_report

number_of_rows                 62
missing_values                  2
exact_duplicate_rows            1
duplicated_transaction_ids      2
invalid_or_missing_amounts      2
non_positive_amounts            1
invalid_timestamps              1
non_standard_timestamps         1
invalid_account_references      1
invalid_merchant_references     1
Name: count, dtype: int64

## Initial quality findings

The corrupted transactions dataset contains several deliberately introduced problems:

- The dataset contains 62 rows instead of the original 60
- Two cells contain missing values
- One row is an exact duplicate
- Two transaction identifiers are duplicated
- One amount is missing
- One amount contains non numeric text
- One amount is negative
- One timestamp represents an impossible date
- One timestamp uses a non-standard format
- Currency, transaction type and status values contain inconsistent spelling or formatting
- One transaction references a nonexistent account
- One transaction references a nonexistent merchan

<small>_No cleaning decisions have been applied yet; the next step is to define how each problem should be handled_</small>